In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# if os.environ.get("GEMINI_API_KEY"):
#     print("GEMINI_API_KEY is set in the environment.")
if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is set in the environment.")

OPENAI_API_KEY is set in the environment.


In [3]:
from langchain_openai import ChatOpenAI

# LLM = ChatOpenAI(
#     # model_name="gemini-3.5-flash-lite",
#     # openai_api_key=os.getenv("GEMINI_API_KEY"),
#     model_name="gpt-4o-mini",
#     openai_api_key=os.getenv("OPENAI_API_KEY"),
#     base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
# )
LLM = ChatOpenAI(model_name="gpt-5-nano", temperature=0)


## Extract text from PDF

In [7]:
# from langchain_community.document_loaders import PyPDFLoader
# pdf_path = "./Docs/Resume_Python.pdf"
# loader = PyPDFLoader(pdf_path)
# docs = loader.load()
# docs
# doc = docs[0]

In [8]:
from pypdf import PdfReader
from langchain_core.documents import Document

pdf_path = "./Docs/Resume_Python.pdf"
reader = PdfReader(pdf_path)

# Loop through pages and construct standard LangChain Document objects
docs = []
for page_num, page in enumerate(reader.pages):
    text = page.extract_text()
    
    # Create the Document object matching LangChain's exact internal schema
    doc = Document(
        page_content=text,
        metadata={
            "source": pdf_path,
            "page": page_num
        }
    )
    docs.append(doc)

# Display the final list of Document objects
docs


[Document(metadata={'source': './Docs/Resume_Python.pdf', 'page': 0}, page_content='AkashMishra\nLocation: Pune,India\nPhone:+91-9450359964/+91-9511911944\nEmail:akashmishra454@gmail.com\nLinkedIn|GitHub\nProfessionalSummary\nAmbitiousTechnologyArchitectwithover11yearsofexperienceengineeringPython-based\ncloudapplicationsandserverlessarchitectureswithinAWSecosystems.Expertinapplication\nanddatabasedevelopment,specializinginhigh-trafficRESTfulAPIs,complexdatamigration,\nandAI-drivenfeatures.Proventrackrecordofoptimizingsystemperformance,reducing\noperationalcostsbyupto25%,anddirectinghigh-performingcross-functionalteamstodeliver\nscalableenterprisesolutions.\nTechnicalSkills\n● Programming:Python(Expert),SQL,JavaScript,HTML5,CSS3.\n● Web Frameworks:FastAPI,Flask,Django,RestAPI\n● Cloud (AWS):Lambda,Glue,Aurora,RDS,EC2,DynamoDB,Redshift,S3,SNS,\nAthena,APIGateway,Lex,SecretsManager.\n● Tools & Platforms:Docker,Kubernetes,Git,GitLab,JIRA,Confluence,\nOpenAPI/Swagger.\n● Databases:PostgreS

## **STEP 2: ** SPlit the documents into chunks

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=25
)
chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': './Docs/Resume_Python.pdf', 'page': 0}, page_content='AkashMishra\nLocation: Pune,India\nPhone:+91-9450359964/+91-9511911944\nEmail:akashmishra454@gmail.com\nLinkedIn|GitHub\nProfessionalSummary\nAmbitiousTechnologyArchitectwithover11yearsofexperienceengineeringPython-based\ncloudapplicationsandserverlessarchitectureswithinAWSecosystems.Expertinapplication\nanddatabasedevelopment,specializinginhigh-trafficRESTfulAPIs,complexdatamigration,\nandAI-drivenfeatures.Proventrackrecordofoptimizingsystemperformance,reducing'),
 Document(metadata={'source': './Docs/Resume_Python.pdf', 'page': 0}, page_content='operationalcostsbyupto25%,anddirectinghigh-performingcross-functionalteamstodeliver\nscalableenterprisesolutions.\nTechnicalSkills\n● Programming:Python(Expert),SQL,JavaScript,HTML5,CSS3.\n● Web Frameworks:FastAPI,Flask,Django,RestAPI\n● Cloud (AWS):Lambda,Glue,Aurora,RDS,EC2,DynamoDB,Redshift,S3,SNS,\nAthena,APIGateway,Lex,SecretsManager.\n● Tools & Platforms

In [10]:
len(chunks)

12

## Step 3: Create embeddings for the chunks using the LLM

In [11]:
# from langchain_google_genai import GoogleGenerativeAIEmbeddings

# embedding_model = GoogleGenerativeAIEmbeddings(
#     model="gemini-embedding-2",
#     # google_api_key="your_key_here"  # Optional: Or use the GOOGLE_API_KEY environment variable
# )

from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    # openai_api_key=os.getenv("OPENAI_API_KEY")  # Optional: Or use the OPENAI_API_KEY environment variable
)

## Step4: Create and Store Embeddings in Vector Store

In [12]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    # persist_directory="chroma_db"
    )


In [13]:
context = vectorstore.similarity_search("candidate all worked companies", k=10)

In [15]:
response = LLM.invoke("How many companies candidate worked in total with time period?" + str(context))
# response = LLM.invoke("How many companies candidate worked in total?")
print(response.content)

Total companies: 4

- Tata Consultancy Services — 03/2015–11/2019
- Accenture — 11/2019–05/2020
- Larsen & Toubro — 05/2020–03/2025
- Wipro — 04/2025–Present
